In [4]:
import pandas as pd

monolith_path = "/Users/khalsel/Documents/projects/MonoRefact/MonoRefact/data/refacteval-final/testing-apps/reports/parsed_reports_monoliths.csv"
microservices_path = "/Users/khalsel/Documents/projects/MonoRefact/MonoRefact/data/refacteval-final/testing-apps/reports/parsed_reports_monomorph.csv"

df_mono = pd.read_csv(monolith_path)
df_monomorph = pd.read_csv(microservices_path)

In [5]:
mono_tests = df_mono.apply(lambda row: f"{row['app']}::{row['test_class']}::{row['test_method']}", axis=1).tolist()
df_mono_indexed = df_mono.copy()
df_mono_indexed["test_fqn"] = mono_tests
df_mono_indexed.set_index("test_fqn", inplace=True)
mm_tests = df_monomorph.apply(lambda row: f"{row['app']}::{row['test_class']}::{row['test_method']}", axis=1).tolist()
df_monomorph["test_fqn"] = mm_tests

In [12]:
rows = []

for app in df_monomorph["app"].unique():
    dft = df_monomorph[df_monomorph.app==app]
    for ms in dft["ms_name_or_monolith"].unique():
        dft2 = dft[(dft.ms_name_or_monolith==ms)]
        count = len(dft2)
        mono_dft2 = df_mono_indexed.loc[dft2["test_fqn"]]
        matching_tests_index = dft2.apply(lambda row: row["status"]==mono_dft2.loc[row["test_fqn"], "status"], axis=1)
        matching_tests_count = matching_tests_index.sum()
        passed_tests = dft2[dft2.status == "passed"]
        mono_passed_tests = df_mono_indexed.loc[passed_tests["test_fqn"]]
        passed_count = len(passed_tests)
        total_test_count = len(dft2)
        total_time = passed_tests["time"].sum()
        avg_time = passed_tests["time"].mean() if passed_count > 0 else 0
        mono_total_time = mono_passed_tests["time"].sum()
        mono_avg_time = mono_passed_tests["time"].mean() if passed_count > 0 else 0
        rows.append({
            "app": app,
            "ms_name_or_monolith": ms,
            "matching_tests_count": matching_tests_count,
            "passed_tests_count": passed_count,
            "total_test_count": total_test_count,
            "fpr": matching_tests_count / total_test_count if total_test_count > 0 else 0,
            "passed_total_time": total_time,
            "passed_avg_time": avg_time,
            "passed_mono_total_time": mono_total_time,
            "passed_mono_avg_time": mono_avg_time,
        })
df_summary = pd.DataFrame(rows, columns=["app", "ms_name_or_monolith", "matching_tests_count", "passed_tests_count", "total_test_count", "fpr", "passed_total_time", "passed_avg_time", "passed_mono_total_time", "passed_mono_avg_time"])
df_summary

,app,ms_name_or_monolith,matching_tests_count,passed_tests_count,total_test_count,fpr,passed_total_time,passed_avg_time,passed_mono_total_time,passed_mono_avg_time
0,spring-petclinic,customers,23,22,33,0.696970,0.899,0.040864,0.542,0.024636
1,spring-petclinic,visits,2,1,5,0.400000,0.388,0.388000,0.003,0.003000
2,spring-petclinic,vets,5,4,15,0.333333,0.806,0.201500,0.079,0.019750
3,jpetstore-6,order,27,27,27,1.000000,0.973,0.036037,0.211,0.007815
4,jpetstore-6,catalog,29,29,29,1.000000,0.879,0.030310,0.092,0.003172
5,jpetstore-6,account,18,18,18,1.000000,0.862,0.047889,0.738,0.041000
6,7ep-demo,mathematics,73,67,73,1.000000,1.076,0.016060,0.211,0.003149
7,7ep-demo,library,94,94,94,1.000000,1.325,0.014096,0.392,0.004170
8,7ep-demo,persistence,32,32,32,1.000000,1.060,0.033125,0.195,0.006094
9,7ep-demo,authentication,61,57,61,1.000000,2.239,0.039281,1.494,0.026211


In [16]:
df_summary = df_summary[["app", "ms_name_or_monolith", "matching_tests_count", "passed_tests_count", "total_test_count", "fpr"]]
overall_rows = []
for app in df_summary["app"].unique():
    dft = df_summary[df_summary.app==app]
    overall_rows.append({
        "app": app,
        "ms_name_or_monolith": "Overall",
        "matching_tests_count": dft["matching_tests_count"].sum(),
        "passed_tests_count": dft["passed_tests_count"].sum(),
        "total_test_count": dft["total_test_count"].sum(),
        "fpr": dft["matching_tests_count"].sum() / dft["total_test_count"].sum() if dft["total_test_count"].sum() > 0 else 0,
    })
all_overall_row = {
    "app": "Overall",
    "ms_name_or_monolith": "Overall",
    "matching_tests_count": df_summary["matching_tests_count"].sum(),
    "passed_tests_count": df_summary["passed_tests_count"].sum(),
    "total_test_count": df_summary["total_test_count"].sum(),
    "fpr": df_summary["matching_tests_count"].sum() / df_summary["total_test_count"].sum() if df_summary["total_test_count"].sum() > 0 else 0,
}
overall_rows.append(all_overall_row)
df_overall_summary = pd.concat([
    df_summary,
    pd.DataFrame(overall_rows, columns=df_summary.columns),
], ignore_index=True)
# sort by app and ms_name_or_monolith but the overall row should be at the end of each app and then the overall row at the end
df_overall_summary["app"] = df_overall_summary["app"].astype(str)
df_overall_summary["ms_name_or_monolith"] = df_overall_summary["ms_name_or_monolith"].astype(str)
df_overall_summary["app"] = df_overall_summary["app"].apply(lambda x: x if x != "Overall" else "zZZ_Overall")
df_overall_summary["ms_name_or_monolith"] = df_overall_summary["ms_name_or_monolith"].apply(lambda x: x if x != "Overall" else "zZZ_Overall")
df_overall_summary = df_overall_summary.sort_values(["app", "ms_name_or_monolith"])
df_overall_summary["app"] = df_overall_summary["app"].apply(lambda x: x if x != "zZZ_Overall" else "Overall")
df_overall_summary["ms_name_or_monolith"] = df_overall_summary["ms_name_or_monolith"].apply(lambda x: x if x != "zZZ_Overall" else "Overall")
# sort by app
df_overall_summary

,app,ms_name_or_monolith,matching_tests_count,passed_tests_count,total_test_count,fpr
9,7ep-demo,authentication,61,57,61,1.000000
7,7ep-demo,library,94,94,94,1.000000
6,7ep-demo,mathematics,73,67,73,1.000000
8,7ep-demo,persistence,32,32,32,1.000000
12,7ep-demo,Overall,260,250,260,1.000000
5,jpetstore-6,account,18,18,18,1.000000
4,jpetstore-6,catalog,29,29,29,1.000000
3,jpetstore-6,order,27,27,27,1.000000
11,jpetstore-6,Overall,74,74,74,1.000000
0,spring-petclinic,customers,23,22,33,0.696970


In [24]:
# Format the DataFrame to display as a table in latex

df_overall_summary_latex = df_overall_summary.copy()
df_overall_summary_latex = df_overall_summary_latex.rename(columns={
    "matching_tests_count": "Matching Tests",
    "passed_tests_count": "Passed Tests",
    "total_test_count": "Total Tests",
    "fpr": "FPR",
    "ms_name_or_monolith": "Microservice",
    "app": "Application",
})
df_overall_summary_latex.replace(
    {
        'Application': {
            'spring-petclinic': 'PetClinic',
            'jpetstore-6': 'JPetStore',
            '7ep-demo': '7ep Demo',
        }
    },
    inplace=True
)
df_overall_summary_latex["Microservice"] = df_overall_summary_latex["Microservice"].apply(lambda x: x.title())
df_overall_summary_latex = df_overall_summary_latex.set_index(['Application', 'Microservice'])
df_overall_summary_latex["FPR"] = df_overall_summary_latex["FPR"].apply(lambda x: f"{x*100:.1f}\\%")
df_overall_summary_latex = df_overall_summary_latex.style.format({
    'Matching Tests': '{:,.0f}',
    'Passed Tests': '{:,.0f}',
    'Total Tests': '{:,.0f}',
    # 'FPR': '{:.1%}',
})
df_overall_summary_latex

In [25]:
df_overall_summary_latex_str = df_overall_summary_latex.to_latex(
    hrules=True,
    label="tab-mm:test_results_monomorph",
    caption="Test Outcome Equivalence Results for Monomorph Microservices",
    position="htbp",
    column_format="ll|cccc",
)
print(df_overall_summary_latex_str)

\begin{table}[htbp]
\caption{Test Outcome Equivalence Results for Monomorph Microservices}
\label{tab-mm:test_results_monomorph}
\begin{tabular}{ll|cccc}
\toprule
{} & {} & {Matching Tests} & {Passed Tests} & {Total Tests} & {FPR} \\
{Application} & {Microservice} & {} & {} & {} & {} \\
\midrule
\multirow[c]{5}{*}{7ep Demo} & Authentication & 61 & 57 & 61 & 100.0\% \\
 & Library & 94 & 94 & 94 & 100.0\% \\
 & Mathematics & 73 & 67 & 73 & 100.0\% \\
 & Persistence & 32 & 32 & 32 & 100.0\% \\
 & Overall & 260 & 250 & 260 & 100.0\% \\
\multirow[c]{4}{*}{JPetStore} & Account & 18 & 18 & 18 & 100.0\% \\
 & Catalog & 29 & 29 & 29 & 100.0\% \\
 & Order & 27 & 27 & 27 & 100.0\% \\
 & Overall & 74 & 74 & 74 & 100.0\% \\
\multirow[c]{4}{*}{PetClinic} & Customers & 23 & 22 & 33 & 69.7\% \\
 & Vets & 5 & 4 & 15 & 33.3\% \\
 & Visits & 2 & 1 & 5 & 40.0\% \\
 & Overall & 30 & 27 & 53 & 56.6\% \\
Overall & Overall & 364 & 351 & 387 & 94.1\% \\
\bottomrule
\end{tabular}
\end{table}



In [22]:
failed_rows = []

for app in df_monomorph["app"].unique():
    dft = df_monomorph[df_monomorph.app==app]
    for ms in dft["ms_name_or_monolith"].unique():
        dft2 = dft[(dft.ms_name_or_monolith==ms)]
        count = len(dft2)
        mono_dft2 = df_mono_indexed.loc[dft2["test_fqn"]]
        non_matching_tests_index = ~dft2.apply(lambda row: row["status"]==mono_dft2.loc[row["test_fqn"], "status"], axis=1)
        failed_rows.append(dft2[non_matching_tests_index])
df_non_matching = pd.concat(failed_rows, ignore_index=True)
df_non_matching

,app,ms_name_or_monolith,test_class,test_method,status,time,test_fqn
0,spring-petclinic,customers,org.springframework.samples.petclinic.service....,shouldFindVets,errored,0.000,spring-petclinic::org.springframework.samples....
1,spring-petclinic,customers,org.springframework.samples.petclinic.service....,shouldFindOwnersByLastName,errored,0.001,spring-petclinic::org.springframework.samples....
2,spring-petclinic,customers,org.springframework.samples.petclinic.service....,shouldAddNewVisitForPet,errored,0.000,spring-petclinic::org.springframework.samples....
3,spring-petclinic,customers,org.springframework.samples.petclinic.service....,shouldUpdateOwner,errored,0.001,spring-petclinic::org.springframework.samples....
4,spring-petclinic,customers,org.springframework.samples.petclinic.service....,shouldFindVisitsByPetId,errored,0.001,spring-petclinic::org.springframework.samples....
5,spring-petclinic,customers,org.springframework.samples.petclinic.service....,shouldInsertPetIntoDatabaseAndGenerateId,errored,0.001,spring-petclinic::org.springframework.samples....
6,spring-petclinic,customers,org.springframework.samples.petclinic.service....,shouldInsertOwner,errored,0.000,spring-petclinic::org.springframework.samples....
7,spring-petclinic,customers,org.springframework.samples.petclinic.service....,shouldFindSingleOwnerWithPet,errored,0.001,spring-petclinic::org.springframework.samples....
8,spring-petclinic,customers,org.springframework.samples.petclinic.service....,shouldUpdatePetName,errored,0.001,spring-petclinic::org.springframework.samples....
9,spring-petclinic,customers,org.springframework.samples.petclinic.service....,shouldFindAllPetTypes,errored,0.000,spring-petclinic::org.springframework.samples....


In [23]:
df_non_matching["test_class"].unique()

array(['org.springframework.samples.petclinic.service.ClinicServiceTests',
       'org.springframework.samples.petclinic.owner.VisitControllerTests'],
      dtype=object)